<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [2]:
from datetime import datetime
import geopandas as gpd
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

In [18]:
initials = "nm"
site = "fiji"
filename = f"{site}-tdata.csv"
version = f"{initials}-lulc-{site}"

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard

In [9]:
joined_df = gpd.read_file(f"training-data/{filename}")
joined_df = joined_df.astype("float32")
joined_df
# joined_df = gpd.read_file("training-data/nm-tongatapu-11122025_postcard_4-training.csv")

,LULC_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,2.0,0.3039,0.0320,0.0348,0.0527,0.088335,2.007067e-07,0.000007,0.3482,0.3543,...,0.443684,3.317686,0.482201,-0.308222,0.308222,-0.532579,-0.512525,-0.063260,-2003285.0,3126635.0
1,2.0,0.3039,0.0320,0.0348,0.0527,0.088335,2.007067e-07,0.000007,0.3482,0.3543,...,0.443684,3.317686,0.482201,-0.308222,0.308222,-0.532579,-0.512525,-0.063260,-2003285.0,3126635.0
2,1.0,0.3787,0.0250,0.0221,0.0461,0.060455,1.397848e-07,0.000003,0.3850,0.3476,...,0.467715,3.969602,0.584718,-0.360273,0.360273,-0.574543,-0.677025,-0.088092,-2003285.0,3062525.0
3,2.0,0.2388,0.0556,0.0345,0.0567,0.075399,1.768053e-07,0.000005,0.2991,0.3095,...,0.528530,1.971924,0.370685,-0.071813,0.071813,-0.247376,-0.464825,-0.071812,-2003285.0,3078095.0
4,1.0,0.3316,0.0478,0.0385,0.0672,0.072500,1.812935e-07,0.000005,0.3333,0.2994,...,0.517431,2.843911,0.418491,-0.206696,0.206696,-0.394762,-0.589700,-0.087329,-2003285.0,3079515.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11243,1.0,0.3502,0.0260,0.0250,0.0474,0.073701,5.921721e-08,0.000004,0.3905,0.3936,...,0.450245,3.831510,0.557070,-0.311365,0.311365,-0.550413,-0.636950,-0.090449,-2003285.0,3087245.0
11244,1.0,0.4319,0.0219,0.0247,0.0435,0.132331,6.009936e-08,0.000007,0.4225,0.4016,...,0.423375,5.551414,0.560682,-0.440627,0.440627,-0.661874,-0.748200,-0.094337,-2003285.0,3087315.0
11245,1.0,0.4296,0.0206,0.0220,0.0441,0.066965,7.606148e-08,0.000004,0.4466,0.4440,...,0.426810,5.323420,0.593287,-0.394579,0.394579,-0.636825,-0.772000,-0.106997,-2003285.0,3087525.0
11246,1.0,0.4260,0.0248,0.0254,0.0493,0.079729,1.176059e-07,0.000004,0.4038,0.4133,...,0.432984,4.976635,0.550725,-0.425703,0.425703,-0.639964,-0.729175,-0.088576,-2003285.0,3086685.0


In [10]:
joined_df

,LULC_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,2.0,0.3039,0.0320,0.0348,0.0527,0.088335,2.007067e-07,0.000007,0.3482,0.3543,...,0.443684,3.317686,0.482201,-0.308222,0.308222,-0.532579,-0.512525,-0.063260,-2003285.0,3126635.0
1,2.0,0.3039,0.0320,0.0348,0.0527,0.088335,2.007067e-07,0.000007,0.3482,0.3543,...,0.443684,3.317686,0.482201,-0.308222,0.308222,-0.532579,-0.512525,-0.063260,-2003285.0,3126635.0
2,1.0,0.3787,0.0250,0.0221,0.0461,0.060455,1.397848e-07,0.000003,0.3850,0.3476,...,0.467715,3.969602,0.584718,-0.360273,0.360273,-0.574543,-0.677025,-0.088092,-2003285.0,3062525.0
3,2.0,0.2388,0.0556,0.0345,0.0567,0.075399,1.768053e-07,0.000005,0.2991,0.3095,...,0.528530,1.971924,0.370685,-0.071813,0.071813,-0.247376,-0.464825,-0.071812,-2003285.0,3078095.0
4,1.0,0.3316,0.0478,0.0385,0.0672,0.072500,1.812935e-07,0.000005,0.3333,0.2994,...,0.517431,2.843911,0.418491,-0.206696,0.206696,-0.394762,-0.589700,-0.087329,-2003285.0,3079515.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11243,1.0,0.3502,0.0260,0.0250,0.0474,0.073701,5.921721e-08,0.000004,0.3905,0.3936,...,0.450245,3.831510,0.557070,-0.311365,0.311365,-0.550413,-0.636950,-0.090449,-2003285.0,3087245.0
11244,1.0,0.4319,0.0219,0.0247,0.0435,0.132331,6.009936e-08,0.000007,0.4225,0.4016,...,0.423375,5.551414,0.560682,-0.440627,0.440627,-0.661874,-0.748200,-0.094337,-2003285.0,3087315.0
11245,1.0,0.4296,0.0206,0.0220,0.0441,0.066965,7.606148e-08,0.000004,0.4466,0.4440,...,0.426810,5.323420,0.593287,-0.394579,0.394579,-0.636825,-0.772000,-0.106997,-2003285.0,3087525.0
11246,1.0,0.4260,0.0248,0.0254,0.0493,0.079729,1.176059e-07,0.000004,0.4038,0.4133,...,0.432984,4.976635,0.550725,-0.425703,0.425703,-0.639964,-0.729175,-0.088576,-2003285.0,3086685.0


In [11]:
joined_df=joined_df.drop(columns=["y", "x"])

In [12]:
print(len(joined_df.columns))
joined_df.columns

33


Index(['LULC_code', 'nir', 'red', 'blue', 'green', 'emad', 'smad', 'bcmad',
       'nir08', 'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2',
       'rededge3', 'mndwi', 'ndti', 'cai', 'ndvi', 'evi', 'savi', 'ndwi',
       'b_g', 'b_r', 'swir22_swir16', 'mci', 'ndci', 'nbi', 'ndmi', 'bsi',
       'awei', 'tc_wetness'],
      dtype='object')

In [13]:
from sklearn.model_selection import train_test_split

training_data, test_data = train_test_split(joined_df, test_size=0.2, random_state=1337)

# The classes are the first column
classes = np.array(training_data)[:, 0]

# The observation data is everything after the second column
observations = np.array(training_data)[:, 1:]

# Create a model...
classifier = RandomForestClassifier(max_depth=4)

# ...and fit it to the data
model = classifier.fit(observations, classes)

In [20]:
# Dynamically create the filename with f-string
file_path = f"models/{version}-test.model"

# Save the model
joblib.dump(model, file_path)

['models/nm-lulc-fiji-test.model']

In [21]:
import pandas as pd
test_actual = np.array(test_data)[:, 0]

test_predicted = model.predict(np.array(test_data)[:,1:])


unique_labels = sorted(np.unique(np.concatenate([np.asarray(test_actual), np.asarray(test_predicted)])))


pd.crosstab(test_actual, test_predicted, margins=True)

col_0,2.0,3.0,4.0,5.0,7.0,All
row_0,,,,,,
1.0,127,27,7,7,1,169
2.0,965,31,32,6,0,1034
3.0,320,68,34,20,1,443
4.0,295,13,59,10,0,377
5.0,23,7,0,70,0,100
6.0,57,12,1,3,0,73
7.0,11,15,0,17,11,54
All,1798,173,133,133,13,2250


In [22]:
from sklearn.metrics import accuracy_score

accuracy_score(test_actual, test_predicted)

0.5213333333333333